# RAGAS 구현하기
기존 RAG 과정에 ReverseHyDE를 도입 → 어느 정도의 성능 향상이 있는지 RAGAS 평가지표를 기준으로 확인해보고자 했다.

In [ ]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters

In [ ]:
import os

In [ ]:
# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [ ]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
# 8. OpenAI LLM 설정 (GPT-4o 또는 GPT-3.5-turbo 등)
llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

## 토크나이저 정의 및 PDF 문서 로드

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

In [ ]:
# 5. 토크나이저 설정
tokenizer = tiktoken.get_encoding("cl100k_base")   # GPT-4, GPT-3.5-turbo 모델들이 사용하는 인코딩 방식

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [ ]:
!pip install chromadb

In [ ]:
!pip install langchain-chroma

In [ ]:
from langchain_chroma import Chroma

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

## Text Splitting

In [ ]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/drive/MyDrive/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

In [ ]:
tiktoken_length = []
for doc in texts:
    # chunk(Document 객체)가 아니라 내부의 텍스트(.page_content)를 전달
    tiktoken_length.append(tiktoken_len(doc.page_content))

print(f"생성된 조각 개수: {len(texts)}")
print(f"조각별 토큰 길이: {tiktoken_length}")

### 텍스트 임베딩

In [ ]:
import openai

In [ ]:
client = openai.OpenAI()

In [ ]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

In [ ]:
from langchain_openai import OpenAIEmbeddings

## 벡터 데이터베이스(Vector DB) 구축

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
db = Chroma.from_documents(texts, embedding_model)

In [ ]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [ ]:
print(docs[3].page_content)

## Reverse HyDE 구현

In [ ]:
# [핵심 로직] Reverse HyDE 구현 예시
from langchain_core.documents import Document

# 1. 데이터를 담을 빈 리스트 생성
reverse_hyde_texts = []

# 2. 진행 상황 출력
print(f"총 {len(texts)}개의 문서 조각에서 예상 질문 생성을 시작합니다...")

# 3. 루프 실행
for i, doc in enumerate(texts):
    # 진행 상황 출력
    print(f"[{i+1}/{len(texts)}] 번째 조각 처리 중...")

    # LLM에게 예상 질문 생성 요청 (문구는 필요에 따라 조정 가능)
    # response.content는 LLM이 생성한 답변 텍스트입니다.
    response = llm_openai.invoke(f"다음 내용을 읽고, 이 내용이 정답이 될 수 있는 질문 3개만 써줘:\n\n{doc.page_content}")

    # 원본 내용 + 예상 질문 합치기 (이것이 Reverse HyDE의 핵심입니다)
    enhanced_content = f"예상 질문들:\n{response.content}\n\n원본 내용:\n{doc.page_content}"

    # 새로운 Document 객체 생성 (기존 metadata 유지)
    new_doc = Document(page_content=enhanced_content, metadata=doc.metadata)
    reverse_hyde_texts.append(new_doc)

# 4. 강화된 데이터를 벡터 DB에 저장 (루프가 완전히 끝난 후 실행)
# embedding_model은 이미 정의되어 있다고 가정합니다.
docsearch = Chroma.from_documents(
    documents=reverse_hyde_texts,
    embedding=embedding_model,
    collection_name="reverse_hyde_collection" # 선택 사항: 컬렉션 이름 지정
)

print(f"총 {len(reverse_hyde_texts)}개의 강화된 문서로 Vector Database 구축이 완료되었습니다!")

## Retrieval

In [ ]:
!pip install -U langchain langchain-classic

In [ ]:
# from langchain.chains import RetrievalQA에서 바꾼 것
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [ ]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [ ]:
qa = RetrievalQA.from_chain_type(
    llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(
      search_type="mmr",
      search_kwargs={"k": 3, "fetch_k" : 10}),
    return_source_documents=True)

In [ ]:
# 10. 질문 실행 및 결과 출력
query = "Who is Sinclair?"
result = qa.invoke(query)

from IPython.display import Markdown, display
display(Markdown(result["result"]))

#### RAG를 사용하지 않은 llm 호출

In [ ]:
# llm2 = ChatOpenAI(
#     model="gpt-4o-mini")
# request = llm2.invoke("how demian looks like")
# display(Markdown(request.content))

## RAGAS

In [ ]:
!pip install ragas datasets

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
import pandas as pd

In [ ]:
# 1. 평가용 질문 및 정답 정의
eval_questions = [
    "싱클레어는 누구니?",
    "데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?",
    "크로머는 어떤 방식으로 싱클레어를 협박했나요?"
]

ground_truths = [
    "싱클레어는 소설 '데미안'의 주인공으로, 밝은 세계와 어두운 세계 사이에서 방황하며 자아를 찾아가는 인물입니다.",
    "데미안은 카인의 표식이 악의 상징이 아니라, 남들보다 강하고 지적인 사람들을 구별해주는 고귀한 표식이라고 설명했습니다.",
    "크로머는 싱클레어가 이웃집 과수원에서 사과를 훔쳤다는 거짓 고백을 약점 잡아 돈을 요구하며 그를 괴롭혔습니다."
]

In [ ]:
# 2. 결과 수집
results = []
for query in eval_questions:
    # QA 시스템 호출
    response = qa.invoke(query)

    # 답변 추출 (RetrievalQA 결과 구조에 따라 수정)
    answer = response.get("result") or response.get("answer")

    # 검색된 근거 추출 (위에서 만든 docsearch 활용)
    retrieved_docs = docsearch.as_retriever().invoke(query)
    context_list = [doc.page_content for doc in retrieved_docs]

    results.append({
        "question": query,
        "answer": answer,
        "contexts": context_list
    })


In [ ]:
# 3. 데이터셋 변환
data = {
    "question": [r["question"] for r in results],
    "answer": [r["answer"] for r in results],
    "contexts": [r["contexts"] for r in results],
    "ground_truth": ground_truths
}
dataset = Dataset.from_dict(data)

In [ ]:
# 4. Ragas 평가 실행 (OpenAI API 키가 설정되어 있어야 합니다)
print("Ragas 평가를 시작합니다...")
result = evaluate(
    dataset=dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
)

In [ ]:
# 5. 결과 확인
print("\n[Ragas 평가 결과]")
print(result)
df = result.to_pandas()
display(df)

## 정성 평가

In [ ]:
# 1. 분석할 질문 리스트 (평가에 사용했던 질문과 동일하게 설정)
analysis_questions = [
    "싱클레어는 누구니?",
    "데미안이 싱클레어에게 카인과 아벨에 대해 설명한 내용은 무엇인가요?",
    "크로머는 어떤 방식으로 싱클레어를 협박했나요?"
]

print("=== Retrieval Analysis (검색 결과 적절성 검토) ===\n")

analysis_results = []

for i, query in enumerate(analysis_questions):
    print(f"🔎 질문 {i+1}: {query}")

    # docsearch(Chroma DB)에서 관련 문서 검색 (k=3)
    # Reverse HyDE로 구축된 DB이므로 '예상 질문'이 포함된 문서를 가져옵니다.
    retrieved_docs = docsearch.as_retriever(search_kwargs={"k": 3}).invoke(query)

    print(f"✅ 검색된 문서 개수: {len(retrieved_docs)}")

    for j, doc in enumerate(retrieved_docs):
        # 검색된 내용의 앞부분 200자만 추출하여 출력
        content_preview = doc.page_content.replace('\n', ' ')[:200]
        source = doc.metadata.get('source', '알 수 없음')
        page = doc.metadata.get('page', '-')

        print(f"   [문서 {j+1}] (출처: {source}, {page}페이지)")
        print(f"   내용 요약: {content_preview}...")
        print("-" * 50)

    print("\n" + "="*80 + "\n")

# (선택 사항) 결과를 표 형태로 정리해서 보고 싶을 때
analysis_data = []
for query in analysis_questions:
    docs = docsearch.as_retriever(search_kwargs={"k": 3}).invoke(query)
    analysis_data.append({
        "Question": query,
        "Top_1_Context": docs[0].page_content[:300] if docs else "검색 결과 없음",
        "Top_2_Context": docs[1].page_content[:300] if len(docs) > 1 else "-",
        "Top_3_Context": docs[2].page_content[:300] if len(docs) > 2 else "-"
    })

df_analysis = pd.DataFrame(analysis_data)
display(df_analysis)

▶ 위 정성 평가를 보고 우리가 판단해야 하는 것은, <br>
"내가 던진 질문에 대답할 수 있는 진짜 정보(원본 내용)가 검색된 문서들 안에 포함되어 있는가?"이다. <br>

① 원본 내용(Original Content)에 정답이 있는가? <br>
질문과 관련된 내용이 원본 내용에 들어가있는지 눈으로 확인한다.

② 예상 질문(Generated Questions)이 길잡이 역할을 하는가? <br>
Reverse HyDE로 생성된 질문들이 내가 던진 질문과 맥락이 닿아 있는지 본다.

③ 검색 결과의 중복성 (Diversity) <br>
가져온 원본 내용들이 중복된, 같은 문서는 아닌지 확인한다.

### 지표 해석
Context Recall : 1.0

→ 모든 질문에서 1.0(100%)이 나왔다. 즉, Reverse HyDE를 통해 구축한 DB가 질문에 필요한 정답 데이터를 정확하게 찾아오고 있다는 뜻이다.

Context Precision : 0, 0.8, 1.0

→ 2개의 질문에 대한 평가는 높은 편이다. 질문과 밀접한 관련이 있는 문서들이 최상위권으로 검색되고 있다는 뜻이다.

Faithfulness : 0, 1

→ 0점이 2개다. 문서에는 정답이 있는데, 모델이 "모르겠습니다"라고 답했기 때문에 문서 내용을 전혀 활용하지 못했다고 판단.<br>